# Lecția 17 - Crearea Agentiilor AI Locale cu Foundry Local și Qwen

În acest caiet construiți un **asistent de inginerie local** care rulează complet pe stația dvs. de lucru. Nu se folosește inferență în cloud în niciun moment. Asistentul poate:

1. **Apela unelte** prin apeluri de funcții Qwen prin Foundry Local.
2. **Lista și citi fișiere** dintr-un director de proiect izolat.
3. **Analiza cod** cu metrici simple.
4. **Căuta documentație** cu RAG local (Chroma).
5. **Folosi un server MCP local** (sărit frumos dacă nu este configurat niciunul).

Codul agentului arată aproape identic ca în lecțiile din cloud — doar punctul de acces al clientului se mută din cloud pe `localhost`.


## Configurare

Înainte de a rula acest notebook:

1. **Instalați Microsoft Foundry Local** (vedeți [documentația](https://learn.microsoft.com/azure/ai-foundry/foundry-local/) pentru sistemul dumneavoastră de operare).
2. **Descărcați și porniți un model Qwen:**
   ```bash
   foundry model run qwen2.5-7b-instruct
   foundry service status
   ```
3. Instalați pachetele Python de mai jos.

Totul rulează local. O mașină cu ~8 GB RAM este un minim realist.


In [ ]:
%pip install foundry-local-sdk openai chromadb -q

## 1. Conectare la Foundry Local

`FoundryLocalManager` descarcă modelul dacă este necesar, pornește serviciul local și ne oferă un **endpoint compatibil cu OpenAI**. Apoi, indicăm SDK-ul standard OpenAI către acesta. Cheia API este un substitut local — nu este implicată nicio acreditare în cloud.


In [ ]:
from foundry_local import FoundryLocalManager
from openai import OpenAI

MODEL_ALIAS = "qwen2.5-7b-instruct"

# Foundry Local selects the best build for your hardware (CPU / GPU / NPU) automatically.
manager = FoundryLocalManager(MODEL_ALIAS)
model_info = manager.get_model_info(MODEL_ALIAS)

client = OpenAI(
    base_url=manager.endpoint,   # e.g. http://localhost:PORT/v1
    api_key=manager.api_key,     # local placeholder
)

MODEL_ID = model_info.id
print(f"Connected to Foundry Local. Serving: {MODEL_ID}")
print(f"Endpoint: {manager.endpoint}")

In [ ]:
# Quick sanity check: a plain chat completion, running fully on-device.
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is a small language model?"}],
)
print(resp.choices[0].message.content)

## 2. Unelte locale (Operații cu fișiere în sandbox)

Creăm un mic proiect exemplu pe disc, apoi definim unelte pentru rădăcina acelui proiect. Verificarea sandbox contează chiar și local: o unealtă care citește căi arbitrare rulează cu permisiunile utilizatorului tău și poate accesa orice poți tu.


In [ ]:
import json
from pathlib import Path

# Create a tiny sample project so the notebook is self-contained.
PROJECT_ROOT = Path.cwd() / "sample_project"
PROJECT_ROOT.mkdir(exist_ok=True)

(PROJECT_ROOT / "auth.py").write_text(
    '"""Authentication helpers."""\n\n'
    "def login(user, password):\n"
    "    # TODO: hash the password before comparing\n"
    "    return user == 'admin' and password == 'secret'\n\n"
    "def logout(session):\n"
    "    session.clear()\n",
    encoding="utf-8",
)
(PROJECT_ROOT / "utils.py").write_text(
    '"""Utility functions."""\n\n'
    "def clamp(value, low, high):\n"
    "    return max(low, min(value, high))\n",
    encoding="utf-8",
)
print("Sample project created at:", PROJECT_ROOT)

In [ ]:
def _safe_path(path: str) -> Path | None:
    """Resolve a path and confirm it stays inside the project sandbox."""
    full = (PROJECT_ROOT / path).resolve()
    if full == PROJECT_ROOT or PROJECT_ROOT in full.parents:
        return full
    return None


def list_files() -> str:
    """List files in the project directory."""
    files = [p.name for p in PROJECT_ROOT.iterdir() if p.is_file()]
    return ", ".join(files) if files else "(no files)"


def read_file(path: str) -> str:
    """Read a file, but only inside the sandboxed project directory."""
    full = _safe_path(path)
    if full is None:
        return "Access denied: path is outside the project directory."
    if not full.is_file():
        return f"No such file: {path}"
    return full.read_text(encoding="utf-8")


def analyze_code(path: str) -> str:
    """Report simple metrics about a source file."""
    full = _safe_path(path)
    if full is None or not full.is_file():
        return "File not found or access denied."
    text = full.read_text(encoding="utf-8")
    lines = text.splitlines()
    return json.dumps({
        "path": path,
        "lines": len(lines),
        "functions": sum(1 for ln in lines if ln.strip().startswith("def ")),
        "todos": sum(1 for ln in lines if "TODO" in ln or "FIXME" in ln),
    })


print(list_files())

## 3. RAG local cu Chroma

Încorporăm un set mic de fragmente de documentație într-o colecție locală Chroma. Chroma rulează în proces și stochează vectorii pe disc — fără server, fără cloud. Instrumentul `search_docs` recuperează cele mai relevante fragmente pentru o interogare.


In [ ]:
import chromadb

DOCS = {
    "auth": "The login() function checks credentials. It currently compares passwords in plain text, which should be hashed.",
    "sessions": "Sessions are cleared on logout via session.clear(). Sessions are stored in memory and lost on restart.",
    "utils": "clamp(value, low, high) constrains a number to a range. Used throughout the UI layer for bounds checking.",
    "style": "This project follows PEP 8. Functions use snake_case and modules include a docstring at the top.",
}

# Chroma ships with a local default embedding model, so embedding stays on-device.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("project_docs")
collection.upsert(
    ids=list(DOCS.keys()),
    documents=list(DOCS.values()),
)


def search_docs(query: str) -> str:
    """Search the local documentation index for relevant snippets."""
    results = collection.query(query_texts=[query], n_results=2)
    docs = results.get("documents", [[]])[0]
    return "\n".join(docs) if docs else "No relevant documentation found."


print(search_docs("how are passwords handled?"))

## 4. Bucla de apelare a instrumentului

Acum înregistrăm instrumentele cu modelul folosind schema OpenAI pentru instrumente și rulăm bucla standard de apelare a instrumentelor — modelul solicită un instrument, noi îl executăm local, introducem rezultatul înapoi și repetăm până când modelul produce un răspuns final. Apelarea funcției fiabilă a lui Qwen este ceea ce face ca acest proces să funcționeze pe dispozitiv.


In [ ]:
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files", "description": "List files in the project directory.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file inside the project directory.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File name, e.g. auth.py"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "analyze_code", "description": "Report line count, function count and TODO count for a file.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "search_docs", "description": "Search local documentation for a query.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"]},
    }},
]

TOOL_IMPL = {
    "list_files": list_files,
    "read_file": read_file,
    "analyze_code": analyze_code,
    "search_docs": search_docs,
}

SYSTEM_PROMPT = (
    "You are a local engineering assistant. Use the provided tools to inspect the project "
    "and its documentation. Prefer calling a tool over guessing. Be concise."
)

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    """Standard tool-calling loop, running entirely against the local model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content or "(no answer)"

        # Record the assistant's tool-call request.
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })

        # Execute each requested tool locally and feed results back.
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_IMPL[name](**args) if name in TOOL_IMPL else f"Unknown tool: {name}"
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    return "Stopped: reached max tool-calling iterations."


print("Agent ready.")

In [ ]:
# A file-reading question.
print(run_agent("What does auth.py do, and is there anything to fix in it?"))

In [ ]:
# A RAG question.
print(run_agent("According to the docs, how are passwords currently handled?"))

In [ ]:
# A code-analysis question.
print(run_agent("How many functions and TODOs are in auth.py?"))

## 5. MCP local (Opțional)

MCP este un transport, nu un serviciu cloud — un server MCP poate rula ca un proces local prin `stdio`. Celula de mai jos arată cum te-ai conecta la un server MCP local dacă ai unul configurat (de exemplu un server de sistem de fișiere). Sară elegant peste execuție când `LOCAL_MCP_COMMAND` nu este setat, deci notebook-ul încă se execută complet fără el.

Notă de securitate: un server MCP local rulează cu permisiunile utilizatorului tău. Limitează-l la un director de proiect și validează rezultatele sale înainte de a acționa pe baza lor.


In [ ]:
import os

LOCAL_MCP_COMMAND = os.getenv("LOCAL_MCP_COMMAND")  # e.g. "npx -y @modelcontextprotocol/server-filesystem ./sample_project"

if not LOCAL_MCP_COMMAND:
    print("No LOCAL_MCP_COMMAND set — skipping the MCP demo. "
          "Set it to a local MCP server command to try this section.")
else:
    import asyncio
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    async def list_mcp_tools(command: str):
        parts = command.split()
        params = StdioServerParameters(command=parts[0], args=parts[1:])
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [t.name for t in tools.tools]

    names = await list_mcp_tools(LOCAL_MCP_COMMAND)
    print("Local MCP server exposes tools:", names)

## Rezumat

Ai construit un asistent de inginerie care rulează în întregime pe mașina ta:

- **Foundry Local** a servit un model **Qwen** în spatele unui endpoint compatibil cu OpenAI — astfel codul agentului corespunde lecțiilor din cloud.
- **Unelte sandboxed** au oferit agentului acces la fișiere și analiză de cod fără a ieși din directorul proiectului.
- **Chroma** a furnizat **RAG local** peste documentație.
- **MCP local** a arătat cum să refolosești ecosistemul MCP offline.

Nicio inferență în cloud nu a fost folosită în niciun moment.

### Provocare

Extinde agentul local pentru a:

1. **Lucra cu mai multe servere MCP** — conectează un server de sistem de fișiere și un server git și lasă agentul să aleagă între ele.
2. **Folosi memorie locală** — păstrează o scurtă istorie a conversației pe disc ca asistentul să își amintească schimbările anterioare chiar și după repornirea notebook-ului.
3. **Susține orchestrarea multi-agent locală** — adaugă un al doilea agent local (de exemplu un recenzor) și fă ca cei doi să colaboreze la o sarcină.

În următoarea lecție vei învăța cum să securizezi agenții AI implementați.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare a responsabilității**:
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). În timp ce ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea profesională realizată de un om. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări greșite care decurg din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
